# Evaluate a trained VLG-CBM -- Colab version

Same four sections as `evaluate_vlgcbm.ipynb`, adapted to run without the NRP pod: pulls
code from GitHub and data from Google Drive instead of the PVC.

**Before running:** build `colab_package.tar.gz` on the pod with
`scripts/package_for_colab.sh <model> [model...]`, upload it to Drive, and set `DATA_ROOT`
below to wherever you put it (e.g. `MyDrive/vlgcbm`). It should contain the tarball as-is,
or already extracted -- both are handled.

Runtime: use a GPU (Runtime -> Change runtime type -> T4 is enough).

In [ ]:
!pip install -q open_clip_torch ftfy regex loguru "setuptools<81"
# clip/clip.py (vendored in the repo) imports pkg_resources, which setuptools>=81 removed

In [ ]:
import os
REPO_DIR = "/content/VLG-CBM"
if not os.path.isdir(REPO_DIR):
    !git clone -q -b bioclip-birds525 https://github.com/aygovind/VLG-CBM.git {REPO_DIR}
%cd {REPO_DIR}

## Data

Mounts Drive and links the package contents into the layout `vlgcbm_analysis.py` expects.
`concept_files/` already came from the git clone above and needs nothing here.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_ROOT = "/content/drive/MyDrive/vlgcbm"   # <-- where you uploaded colab_package.tar.gz

import glob
tarballs = glob.glob(f"{DATA_ROOT}/*.tar.gz")
if tarballs and not os.path.isdir(f"{DATA_ROOT}/colab_package"):
    !tar xzf {tarballs[0]} -C {DATA_ROOT}
PKG = f"{DATA_ROOT}/colab_package"

os.environ["DATASET_FOLDER"] = "/content/VLG-CBM/datasets"
os.environ["VLGCBM_BIOCLIP_CKPT"] = f"{PKG}/models/bioclip/open_clip_pytorch_model.bin"

!mkdir -p datasets/birds525 annotations saved_models
!ln -sfn {PKG}/data/birds525/val datasets/birds525/val
!ln -sfn {PKG}/annotations/birds525_val annotations/birds525_val
!for d in {PKG}/saved_models/*/; do ln -sfn "$d" "saved_models/$(basename $d)"; done

In [ ]:
import torch
import vlgcbm_analysis as va

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("models:", ", ".join(va.list_models("birds525")))

In [ ]:
MODEL   = "bioclip"      # <-- swap me
DATASET = "birds525"
SPLIT   = "val"

run = va.load_run(va.run_dir(MODEL, DATASET))
res = va.evaluate(run, split=SPLIT)          # cached to <run>/eval_val.pt
print(run)
print("accuracy: {:.2f}%".format(res.accuracy * 100))

## 1. Sankey: concept -> class

In [ ]:
best, worst = va.best_worst_classes(run, res, k=5)
print("best :", best)
print("worst:", worst)

In [ ]:
va.sankey_static(run, [best[0], worst[0]],
                 weight_cutoff=0.05, max_per_class=12,
                 save_path="figures/sankey.png")

## 2. Single example

In [ ]:
CLASS = worst[0]

idx_all   = va.class_indices(res, CLASS)
idx_wrong = va.class_indices(res, CLASS, only="wrong")
print(f"{CLASS}: {len(idx_all)} images, {len(idx_wrong)} wrong")

In [ ]:
IDX = int(idx_wrong[0]) if len(idx_wrong) else int(idx_all[0])
va.explain_example(run, idx=IDX, split=SPLIT)

## 3. Same example, with boxes

In [ ]:
va.explain_with_boxes(run, idx=IDX, split=SPLIT, top_k=8)

## 4. Model comparison grid

Only includes models whose run directory you packaged.

In [ ]:
outs = va.evaluate_many(va.find_runs(), split=SPLIT, keep_concept_acts=True)
for o in outs:
    print(o)

In [ ]:
va.story_figure(outs, split=SPLIT, top_concepts=2,
                flag_mode="sufficiency",
                save_path="figures/qualitative_comparison.png")